**Exercício 1 — Fazer a requisição e inspecionar o HTML**






**Objetivo:** Conectar-se à página e imprimir o HTML bruto.

- Importe `requests` e `BeautifulSoup`.
- Faça um `GET` para `https://www.scrapethissite.com/pages/simple/`.
- Verifique o `status_code` da resposta.
- Imprima os primeiros **500 caracteres** do HTML recebido.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import csv
import time
import sqlite3

In [ ]:
url = "https://www.scrapethissite.com/pages/simple/"
response = requests.get(url, timeout=10)
response.raise_for_status()
sopadeLetrinhas = BeautifulSoup(response.text, "html.parser")
html = response.text

In [ ]:
print("Status code:", response.status_code)

Status code: 200


In [ ]:
print("\nPrimeiros 500 caracteres do HTML:\n")
print(html[:500])


Primeiros 500 caracteres do HTML:

<!doctype html>
<html lang="en">
  <head>
    <meta charset="utf-8">
    <title>Countries of the World: A Simple Example | Scrape This Site | A public sandbox for learning web scraping</title>
    <link rel="icon" type="image/png" href="/static/images/scraper-icon.png" />

    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta name="description" content="A single page that lists information about all the countries in the world. Good for those just get started with w


**Exercício 2 — Extrair os nomes de todos os países**

**Objetivo:** Usar `find_all()` ou `select()` para coletar elementos repetidos.

- Inspecione o HTML da página e identifique a tag/classe que envolve o nome de cada país.
- Extraia **todos os nomes** e armazene em uma lista Python.
- Imprima a lista e confirme que há **250 países**.

In [ ]:
paises = sopadeLetrinhas.select(".country-name")
nomespaises = [c.get_text(strip=True) for c in paises]


In [ ]:
print(nomespaises)

['Andorra', 'United Arab Emirates', 'Afghanistan', 'Antigua and Barbuda', 'Anguilla', 'Albania', 'Armenia', 'Angola', 'Antarctica', 'Argentina', 'American Samoa', 'Austria', 'Australia', 'Aruba', 'Åland', 'Azerbaijan', 'Bosnia and Herzegovina', 'Barbados', 'Bangladesh', 'Belgium', 'Burkina Faso', 'Bulgaria', 'Bahrain', 'Burundi', 'Benin', 'Saint Barthélemy', 'Bermuda', 'Brunei', 'Bolivia', 'Bonaire', 'Brazil', 'Bahamas', 'Bhutan', 'Bouvet Island', 'Botswana', 'Belarus', 'Belize', 'Canada', 'Cocos [Keeling] Islands', 'Democratic Republic of the Congo', 'Central African Republic', 'Republic of the Congo', 'Switzerland', 'Ivory Coast', 'Cook Islands', 'Chile', 'Cameroon', 'China', 'Colombia', 'Costa Rica', 'Cuba', 'Cape Verde', 'Curacao', 'Christmas Island', 'Cyprus', 'Czech Republic', 'Germany', 'Djibouti', 'Denmark', 'Dominica', 'Dominican Republic', 'Algeria', 'Ecuador', 'Estonia', 'Egypt', 'Western Sahara', 'Eritrea', 'Spain', 'Ethiopia', 'Finland', 'Fiji', 'Falkland Islands', 'Micron

In [ ]:
print("\nTotal de países:", len(nomespaises))


Total de países: 250



**Exercício 3 — Extrair capital, população e área**


**Objetivo:** Coletar múltiplos campos de um mesmo elemento.

- Para cada país, extraia também:
    - **Capital** (`country-capital`)
    - **População** (`country-population`)
    - **Área** (`country-area`)
- Armazene cada país como um **dicionário** Python.
- Imprima os 5 primeiros países como teste.

In [ ]:
paisesCPA = sopadeLetrinhas.select(".country")

data = []
for pais in paisesCPA:
    name = pais.select_one(".country-name").get_text(strip=True)
    capital = pais.select_one(".country-capital").get_text(strip=True)
    population = pais.select_one(".country-population").get_text(strip=True)
    area = pais.select_one(".country-area").get_text(strip=True)

    country_dict = {
        "name": name,
        "capital": capital,
        "population": population,
        "area": area
    }

    data.append(country_dict)

for item in data[:5]:
    for chave, valor in item.items():
        print(f"{chave}: {valor}")
    print()


name: Andorra
capital: Andorra la Vella
population: 84000
area: 468.0

name: United Arab Emirates
capital: Abu Dhabi
population: 4975593
area: 82880.0

name: Afghanistan
capital: Kabul
population: 29121286
area: 647500.0

name: Antigua and Barbuda
capital: St. John's
population: 86754
area: 443.0

name: Anguilla
capital: The Valley
population: 13254
area: 102.0



**Exercício 4 — Salvar os dados em CSV**

**Objetivo:** Persistir os dados extraídos em um arquivo.

- Use o módulo `csv` (ou `pandas`) para salvar os dados dos 250 países em um arquivo `paises.csv`.
- O arquivo deve conter os cabeçalhos: `nome`, `capital`, `populacao`, `area_km2`.
- Abra o arquivo gerado e verifique se os dados estão corretos.


**Objetivo:** Persistir os dados extraídos em um arquivo.

- Use o módulo `csv` (ou `pandas`) para salvar os dados dos 250 países em um arquivo `paises.csv`.
- O arquivo deve conter os cabeçalhos: `nome`, `capital`, `populacao`, `area_km2`.
- Abra o arquivo gerado e verifique se os dados estão corretos.

**Objetivo:** Persistir os dados extraídos em um arquivo.

- Use o módulo `csv` (ou `pandas`) para salvar os dados dos 250 países em um arquivo `paises.csv`.
- O arquivo deve conter os cabeçalhos: `nome`, `capital`, `populacao`, `area_km2`.
- Abra o arquivo gerado e verifique se os dados estão corretos.

In [ ]:
with open("paises.csv", "w", newline="", encoding="utf-8") as arquivo:
    writer = csv.DictWriter(
        arquivo,
        fieldnames=["nome", "capital", "populacao", "area_km2"]
    )

    writer.writeheader()

    for item in data:
        writer.writerow({
            "nome": item["name"],
            "capital": item["capital"],
            "populacao": item["population"],
            "area_km2": item["area"]
        })

**Exercício 5 — Filtrar países por população**

**Objetivo:** Combinar extração com lógica de filtragem.

- A partir da lista de países extraída, filtre apenas aqueles com **população acima de 100 milhões**.
- Exiba o nome e a população de cada país filtrado em ordem **decrescente** de população.

In [ ]:
paises_filtrados = []

for item in data:
    populacao_str = item["population"]

    if populacao_str:
        populacao = int(populacao_str.replace(",", ""))

        if populacao > 100000000:
            paises_filtrados.append((item["name"], pop))


paises_filtrados.sort(key=lambda x: x[1], reverse=True)


for nome, populacao in paises_filtrados:
    print(nome+":",populacao)

Bangladesh: 11651858
Brazil: 11651858
China: 11651858
Indonesia: 11651858
India: 11651858
Japan: 11651858
Mexico: 11651858
Nigeria: 11651858
Pakistan: 11651858
Russia: 11651858
United States: 11651858


**Exercício 6 — Encontrar o país com maior e menor área**

**Objetivo:** Aplicar funções de agregação sobre dados raspados.

- Extraia a área de todos os países.
- Descubra e exiba:
    - O país com a **maior área**.
    - O país com a **menor área** (excluindo zeros).

In [ ]:
paises_area = []

for item in data:
    area_str = item["area"]

    if area_str:
        area = float(area_str.replace(",", ""))

        if area > 0:
            paises_area.append((item["name"], area))


maior = max(paises_area, key=lambda x: x[1])

menor = min(paises_area, key=lambda x: x[1])


print("Maior área:")
print(maior[0], maior[1],"km2")

print("\nMenor área:")
print(menor[0], menor[1],"km2")

Maior área:
Russia 17100000.0 km2

Menor área:
Vatican City 0.44 km2


**Exercício 7 — Tratar erros e dados ausentes**

**Objetivo:** Lidar com dados incompletos de forma robusta.

- Alguns países possuem capital listada como `"None"` ou campo vazio.
- Identifique quais países **não têm capital** registrada.
- Substitua esses valores por `"Não informada"` na sua estrutura de dados.

In [ ]:
for item in data:
    capital = item["capital"]

    if not capital or capital == "None":
        print("País sem capital:", item["name"])
        item["capital"] = "Não informada"

País sem capital: Antarctica
País sem capital: Bouvet Island
País sem capital: Heard Island and McDonald Islands
País sem capital: Israel
País sem capital: British Indian Ocean Territory
País sem capital: Palestine
País sem capital: Tokelau
País sem capital: U.S. Minor Outlying Islands


**Exercício 8 — Criar um relatório com pandas**

**Objetivo:** Integrar web scraping com análise de dados.

- Carregue os dados raspados em um **DataFrame** do `pandas`.
- Calcule estatísticas descritivas (média, máximo, mínimo) de `populacao` e `area_km2`.
- Exiba os **10 países mais populosos** em uma tabela formatada.

In [ ]:
df = pd.DataFrame(data)

In [ ]:
df["populacao"] = df["population"].str.replace(",", "").astype(float)
df["area_km2"] = df["area"].str.replace(",", "").astype(float)

In [ ]:
print("População:")
print(df["populacao"].describe())

print("\nÁrea:")
print(df["area_km2"].describe())

População:
count    2.500000e+02
mean     2.744568e+07
std      1.168626e+08
min      0.000000e+00
25%      1.798562e+05
50%      4.288138e+06
75%      1.542062e+07
max      1.330044e+09
Name: populacao, dtype: float64

Área:
count    2.500000e+02
mean     5.996369e+05
std      1.911821e+06
min      0.000000e+00
25%      1.174750e+03
50%      6.489450e+04
75%      3.726315e+05
max      1.710000e+07
Name: area_km2, dtype: float64


In [ ]:
top10 = df.sort_values(by="populacao", ascending=False).head(10)

top10["colocacao"] = range(1, len(top10) + 1)

top10 = top10[["colocacao", "name", "populacao"]]

print(top10.to_string(index=False))

 colocacao          name    populacao
         1         China 1330044000.0
         2         India 1173108018.0
         3 United States  310232863.0
         4     Indonesia  242968342.0
         5        Brazil  201103330.0
         6      Pakistan  184404791.0
         7    Bangladesh  156118464.0
         8       Nigeria  154000000.0
         9        Russia  140702000.0
        10         Japan  127288000.0


Desafio extra (+0,5): Adicione um User-Agent realista ao seu requests.get() e implemente um retry automático com time.sleep() entre tentativas, caso a requisição falhe. Salve o resultado final em um banco de dados SQLite.

In [ ]:


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
}

tentativas = 3

for tentativa in range(tentativas):
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        print("Requisição bem-sucedida!")
        break
    except requests.exceptions.RequestException:
        print(f"Erro na tentativa {tentativa+1}...")
        time.sleep(2)
else:
    print("Falha após várias tentativas.")
    exit()

Requisição bem-sucedida!


In [ ]:
paisesEPA = sopadeLetrinhas.select(".country")

data = []

for country in paisesEPA:
    name = country.select_one(".country-name").get_text(strip=True)
    capital = country.select_one(".country-capital").get_text(strip=True)
    population = country.select_one(".country-population").get_text(strip=True)
    area = country.select_one(".country-area").get_text(strip=True)

    data.append({
        "name": name,
        "capital": capital,
        "population": population,
        "area": area
    })

In [ ]:

df["populacao"] = pd.to_numeric(df["population"].str.replace(",", ""), errors="coerce")
df["area_km2"] = pd.to_numeric(df["area"].str.replace(",", ""), errors="coerce")

In [ ]:

conn = sqlite3.connect("paises.db")

df.to_sql("paises", conn, if_exists="replace", index=False)

conn.close()

print("Dados salvos no banco com sucesso!")

Dados salvos no banco com sucesso!


In [ ]:
conn = sqlite3.connect("paises.db")

df_lido = pd.read_sql("SELECT * FROM paises LIMIT 10", conn)

print(df_lido)

conn.close()

                   name           capital population       area  populacao  \
0               Andorra  Andorra la Vella      84000      468.0      84000   
1  United Arab Emirates         Abu Dhabi    4975593    82880.0    4975593   
2           Afghanistan             Kabul   29121286   647500.0   29121286   
3   Antigua and Barbuda        St. John's      86754      443.0      86754   
4              Anguilla        The Valley      13254      102.0      13254   
5               Albania            Tirana    2986952    28748.0    2986952   
6               Armenia           Yerevan    2968000    29800.0    2968000   
7                Angola            Luanda   13068161  1246700.0   13068161   
8            Antarctica     Não informada          0      1.4E7          0   
9             Argentina      Buenos Aires   41343201  2766890.0   41343201   

     area_km2  
0       468.0  
1     82880.0  
2    647500.0  
3       443.0  
4       102.0  
5     28748.0  
6     29800.0  
7   1246700.0